### Inicialização

In [20]:
import numpy as np
import pandas as pd
import importlib
import src.data.pipeline as data_transform
import src.utils.analyze as data_analyze
import src.models.train as train_models
import src.config as cfg

import joblib
from sklearn.model_selection import train_test_split

importlib.reload(cfg)
importlib.reload(data_transform)
importlib.reload(data_analyze)
importlib.reload(train_models)

cfg.RESULTS_PATH.mkdir(parents=True, exist_ok=True)
cfg.DATA_PATH.mkdir(parents=True, exist_ok=True)

In [21]:
if cfg.RUN_TRANSFORMATION:
    print("A iniciar transformação dos dados brutos...")
    raw_df = pd.read_csv(cfg.RAW_DATA_PATH, sep=',')
    
    # Aplicar transformações
    df = data_transform.transform_data(raw_df)
    
    # Garantir que a pasta 'data' existe e guardar o csv
    df.to_csv(cfg.PROCESSED_FILE_PATH, index=False)
    print(f"Dados transformados e guardados em: {cfg.PROCESSED_FILE_PATH}\n")

else:
    print("A carregar dados já processados da pasta data...")
    df = pd.read_csv(cfg.PROCESSED_FILE_PATH)
    
    # Reconverter as colunas de data (o CSV guarda como string)
    df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
    df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# Correr a análise aos dados finais carregados em memória
data_analyze.analyze_data(df)

A carregar dados já processados da pasta data...

--- Balanceamento da Classe Alvo (No-show) ---
         Contagem  Percentagem (%)
No-show                           
False       88208            79.81
True        22314            20.19
--- Contagem de Valores Nulos ---
PatientId               0
Gender                  0
ScheduledDay            0
AppointmentDay          0
Age                     0
Neighbourhood           0
Scholarship             0
Hipertension            0
Diabetes                0
Alcoholism              0
Handcap                 0
SMS_received            0
No-show                 0
WaitingDays             0
ScheduledHour           0
ScheduledDayOfWeek      0
AppointmentDayOfWeek    0
AppointmentMonth        0
dtype: int64

--- Distribuição das Variáveis Categóricas (%) ---

Gender:
Gender
F    65.0
M    35.0
Name: proportion, dtype: float64

Scholarship:
Scholarship
0    90.17
1     9.83
Name: proportion, dtype: float64

Hipertension:
Hipertension
0    80.27
1    19

In [ ]:
from tqdm.auto import tqdm

print("A processar features...")
X, y = train_models.preprocess_features(df)

# Split inicial para a fase de otimização
X_train_opt, X_test_opt, y_train_opt, y_test_opt = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=42)

modelos = ['LogisticRegression', 'KNN', 'RandomForest', 'XGBoost', 'LightGBM']
best_models_dict = {}

for model_name in tqdm(modelos, desc="Otimização Optuna"):
    tqdm.write(f"\n[{model_name}] A procurar melhores hiperparâmetros...")
    
    # Encontra e guarda o modelo instanciado com os melhores parâmetros
    best_model = train_models.tune_model(X_train_opt, y_train_opt, model_name, n_trials=cfg.OPTUNA_TRAIN_STEPS)
    best_models_dict[model_name] = best_model

print("\n✅ Otimização concluída! Melhores hiperparâmetros encontrados.")

A processar features...


Otimização Optuna:   0%|          | 0/5 [00:00<?, ?it/s]


[LogisticRegression] A procurar melhores hiperparâmetros...


Otimização Optuna:  20%|██        | 1/5 [00:05<00:22,  5.59s/it]


[KNN] A procurar melhores hiperparâmetros...


Otimização Optuna:  40%|████      | 2/5 [00:59<01:41, 33.99s/it]


[RandomForest] A procurar melhores hiperparâmetros...


In [ ]:
import joblib
import numpy as np

# Garantir que a pasta de resultados existe
models_dir = cfg.RESULTS_PATH / "models"
models_dir.mkdir(parents=True, exist_ok=True)

results_list = []
final_trained_models = {}

for model_name, best_model in tqdm(best_models_dict.items(), desc="Treino Final e Avaliação"):
    tqdm.write(f"\n[{model_name}] A testar robustez ({cfg.NUMBER_OF_SEEDS} partições)...")
    
    # Validação Robusta (treina e devolve as métricas e o modelo final)
    metrics, final_model, cm = train_models.evaluate_robustness(X, y, best_model, n_seeds=cfg.NUMBER_OF_SEEDS)
    
    # Guardar os artefactos
    joblib.dump(final_model, models_dir / f"{model_name}_best.pkl")
    train_models.save_confusion_matrix(cm, model_name, models_dir)
    final_trained_models[model_name] = final_model
    
    # Compilar resultados
    res_row = {'Model': model_name}
    for metric_name, values in metrics.items():
        res_row[f"{metric_name}_mean"] = np.mean(values)
        res_row[f"{metric_name}_std"] = np.std(values)
        
    results_list.append(res_row)

# Compilar e guardar a tabela
results_df = pd.DataFrame(results_list)
results_df.to_csv(cfg.RESULTS_PATH / "models_evaluation_results.csv", index=False)

print(f"\nAvaliação concluída! Resultados guardados em: {cfg.RESULTS_PATH}")
display(results_df.sort_values(by='roc_auc_mean', ascending=False).style.background_gradient(cmap='Blues'))

In [ ]:
import shap
import matplotlib.pyplot as plt

# 1. Identificar o modelo vencedor baseado na métrica ROC-AUC
best_model_name = results_df.sort_values(by='roc_auc_mean', ascending=False).iloc[0]['Model']
print(f"O melhor modelo geral foi: {best_model_name}")

vencedor = final_trained_models[best_model_name]

print("\nA calcular valores SHAP... (Isto pode demorar alguns segundos)")

# 2. Configurar o SHAP (TreeExplainer é incrivelmente rápido para Random Forest, XGBoost e LightGBM)
tree_models = ['RandomForest', 'XGBoost', 'LightGBM']

if best_model_name in tree_models:
    explainer = shap.TreeExplainer(vencedor)
    shap_values = explainer.shap_values(X_test_opt)
    
    # Plot 1: Gráfico de Barras (Mostra a importância média absoluta)
    plt.title(f"Top 10 Variáveis Mais Importantes ({best_model_name})")
    shap.summary_plot(shap_values, X_test_opt, plot_type="bar", max_display=10)
    
    # Plot 2: Gráfico de Dispersão (Mostra o impacto na previsão: ex. Idade alta empurra para faltar ou comparecer?)
    plt.title(f"Impacto das Variáveis na Decisão ({best_model_name})")
    shap.summary_plot(shap_values, X_test_opt, max_display=10)

else:
    # Caso o KNN ou Regressão Logística ganhe, usamos um Explainer agnóstico
    explainer = shap.Explainer(vencedor, X_train_opt)
    shap_values = explainer(X_test_opt)
    shap.plots.bar(shap_values, max_display=10)